# 06 — ML Probability Model
**Invariant proved:** ML Model Interface (probability estimation)
**Module built:** `sdk/ml/probability_model.py`

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from sdk.ml.probability_model import ControlledProbabilityModel
from sdk.ml.performance import auc_score, calibration_slope, hosmer_lemeshow_test

## 1. Generate Probabilities

Create true probabilities from a Beta(2, 20) distribution (mean ~0.09) and
generate binary outcomes from those probabilities.

In [ ]:
rng = np.random.default_rng(42)
n = 3000

# True underlying probabilities
true_probs = rng.beta(2, 20, n)

# Generate actual binary outcomes from those probabilities
actuals = (rng.random(n) < true_probs).astype(int)

print(f"N = {n}")
print(f"Mean true probability: {true_probs.mean():.4f}")
print(f"Observed event rate:   {actuals.mean():.4f}")
print(f"Total events:          {actuals.sum()}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(true_probs, bins=50, alpha=0.7, color='steelblue')
ax.set_xlabel('True Probability')
ax.set_ylabel('Count')
ax.set_title('Distribution of True Probabilities — Beta(2, 20)')
ax.axvline(true_probs.mean(), color='red', linestyle='--', label=f'Mean = {true_probs.mean():.3f}')
ax.legend()
plt.tight_layout()
plt.show()

## 2. Fit and Predict

Create a `ControlledProbabilityModel` with target AUC=0.78, fit it to the true
probabilities, and generate predicted probabilities.

In [ ]:
model = ControlledProbabilityModel(target_auc=0.78)

# Fit: find noise parameters that achieve target AUC
model.fit(true_probs, rng)

# Predict: generate calibrated probability estimates
predictions = model.predict(true_probs, rng)

# Evaluate
achieved_auc = auc_score(actuals, predictions)
slope, pred_means, obs_means = calibration_slope(actuals, predictions)
hl_stat, hl_p = hosmer_lemeshow_test(actuals, predictions)

print("=== Achieved Metrics ===")
print(f"AUC:               {achieved_auc:.3f}  (target: 0.78)")
print(f"Calibration Slope: {slope:.3f}  (ideal: 1.0)")
print(f"Hosmer-Lemeshow:   stat={hl_stat:.2f}, p={hl_p:.3f}")
print(f"  (p > 0.05 indicates good calibration)")
print(f"\nPrediction range: [{predictions.min():.4f}, {predictions.max():.4f}]")
print(f"Prediction mean:  {predictions.mean():.4f}")

## 3. Calibration Plot

Plot predicted vs observed means across decile bins. A well-calibrated model
lies along the diagonal (slope near 1.0).

In [ ]:
slope, pred_means, obs_means = calibration_slope(actuals, predictions)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Calibration plot
ax = axes[0]
ax.scatter(pred_means, obs_means, s=80, zorder=5, color='steelblue', edgecolors='navy')
lims = [0, max(pred_means.max(), obs_means.max()) * 1.1]
ax.plot(lims, lims, 'k--', alpha=0.4, label='Perfect calibration')
fit_line = np.polyfit(pred_means, obs_means, 1)
x_fit = np.linspace(lims[0], lims[1], 100)
ax.plot(x_fit, np.polyval(fit_line, x_fit), 'r-', alpha=0.7, label=f'Slope = {slope:.2f}')
ax.set_xlabel('Predicted Probability (bin mean)')
ax.set_ylabel('Observed Rate (bin mean)')
ax.set_title('Calibration Plot — Predicted vs Observed')
ax.legend()
ax.grid(True, alpha=0.3)

# Prediction distribution
ax = axes[1]
ax.hist(predictions[actuals == 0], bins=40, alpha=0.6, label='No event', color='gray')
ax.hist(predictions[actuals == 1], bins=40, alpha=0.7, label='Event', color='red')
ax.set_xlabel('Predicted Probability')
ax.set_ylabel('Count')
ax.set_title('Prediction Distribution by Outcome')
ax.legend()

plt.tight_layout()
plt.show()

print(f"Calibration slope: {slope:.3f}")

## Key Insights

1. **`ControlledProbabilityModel`** generates probability predictions that hit a target AUC by injecting calibrated noise into true probabilities.
2. Unlike the binary classifier (NB05), the probability model targets **discrimination** (AUC) and **calibration** (slope near 1.0) rather than PPV/sensitivity at a threshold.
3. The calibration plot shows predicted vs observed rates across decile bins — a slope near 1.0 means the model's probabilities are trustworthy.
4. The Hosmer-Lemeshow test provides a formal goodness-of-fit check: p > 0.05 indicates the model is well-calibrated.
5. This probability interface is used by scenarios where the ML task is risk estimation (e.g., no-show prediction) rather than binary flag generation.

**Next:** NB07 — Results & AnalysisDataset